In [ ]:
# STRUCTURE:
# actual results, and initial stop (if never moved)
# no stop
# wide stop (20-40% range)
# 8, 10, 12.5, 15% stops
# 2, 2.5, 3 ATR stops
# maybe daily higher low and weekly higher low pivot stops, if I can figure that out, in a more advanced version

# ASSUMPTIONS:
# Entry is taken around the close of the day, so the first day is skipped when assessing R-multiples. Hence df.iloc[1:] syntax.
# On days where price gaps down below my stop, the stop is triggered at the open. Not always going to be the case
# No commissions or slippage on exit, so a round -1R loss if stop is hit intraday. Would like to fix in a future version.
# Intraday action is currently ignored. This may be an issue on days where my stop hits before a new high for the move. Will probably address with intraday price data and resampling. 

# CONSIDERATIONS:
# Think about intraday entry and exit timing. E.g. what if my stop is below the low of the day, but I enter after the low of the day is set? Does it matter if entry is at the close?
# Would I go about this process a different way with a larger dataset? Does pandas have a built in function, so I don't have to use the slower python loops?
# Thinking it might be a good idea to use multiple functions for this. E.g. reading data, running simulation, outputting results.
# While this isn't a consideration for v1, for future versions, I'd like to consider intraday stops, slippage, commissions,etc. I want to make this professional-grade.
# Should I add some kind of drawdown calculation in a future version, so I can see what type of pullbacks I might expect and how viable wide stops actually are. 
# At some point I plan to test trailing stops on profitable trades to see what the most effective method there would be. E.g. pivot lows, moving average, atr, percentage trailing, etc.

# ERRORS:
# rounding issue on exit_price of initial_stop function...

In [39]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

In [ ]:
# Actual trade result function...

def actual_outcome(ticker, entry_date, entry_price, stop_price, exit_date, exit_price):

    data = yf.download(ticker, start=entry_date, multi_level_index=False, auto_adjust=True, progress=False)

    realised_r = (exit_price - entry_price) / (entry_price - stop_price)
    max_price = entry_price
    
    date = pd.to_datetime(entry_date) + pd.DateOffset(days=1)
    
    if not pd.isna(exit_date):
        for row_name, row in data.loc[date:exit_date].iterrows():

            if row['High'] > max_price:
                max_price = row['High']
    else:
        for row_name, row in data.iloc[1:].iterrows():

            if row['High'] > max_price:
                max_price = row['High']

    max_r = (max_price  - entry_price) / (entry_price - stop_price)

    return {'entry_date': entry_date, 
            'ticker': ticker, 
            'entry_price': entry_price,
            'stop_price': stop_price,
            'exit_date': exit_date,
            'exit_price': exit_price,
            'realised_r': round(realised_r, 2), 
            'max_r': round(max_r, 2),
            'stop_type': 'Actual Outcome'}

In [ ]:
# Initial stop result function...

def initial_stop(ticker, entry_date, entry_price, stop_price):

    data = yf.download(ticker, start=entry_date, multi_level_index=False, auto_adjust=True, progress=False)

    max_price = entry_price
    realised_r = np.nan
    exit_date = np.nan
    exit_price = np.nan
    
    for row_name, row in data.iloc[1:].iterrows():

        if row['High'] > max_price:
            max_price = row['High']

        if stop_price >= row['Low']:
            if stop_price >= row['Open']:
                realised_r = (row['Open'] - entry_price) / (entry_price - stop_price)
                exit_price = row['Open']
                
            else:
                realised_r = (stop_price - entry_price) / (entry_price - stop_price)
                exit_price = stop_price
            
            exit_date = row_name.date()
            break
    
    if not pd.isna(realised_r):
        realised_r = round(realised_r, 2)

    max_r = (max_price - entry_price) / (entry_price - stop_price)

    return {'entry_date': entry_date, 
            'ticker': ticker, 
            'entry_price': entry_price, 
            'stop_price': stop_price, 
            'exit_date': exit_date,
            'exit_price': exit_price, # should round exit price, but check pd.isna() first...
            'realised_r': realised_r, 
            'max_r': round(max_r, 2),
            'stop_type': 'Initial Stop'}

# Edge cases and improvements to consider for later: 
# what if there's no data from df.iloc[1:]? 
# what should realised_r return if not stopped out? na value?
# what happens on a day if my stop hits before the high of the day? 

In [44]:
def no_stop(ticker, entry_date, entry_price, stop_price): 

    data = yf.download(ticker, start=entry_date, multi_level_index=False, auto_adjust=True, progress=False)

    max_price = entry_price

    for row_name, row in data.iloc[1:].iterrows():

        if row['High'] > max_price:
            max_price = row['High']

    max_r = (max_price - entry_price) / (entry_price - stop_price)

    return {'entry_date': entry_date, 
            'ticker': ticker, 
            'entry_price': entry_price, 
            'stop_price': stop_price, 
            'exit_date': np.nan,
            'exit_price': np.nan,
            'realised_r': np.nan, 
            'max_r': round(max_r, 2),
            'stop_type': 'No Stop'}

In [45]:
# Reading trades CSV file, then using it as input for a list of dicts, which stores trade outputs.
# Instead of looping yfinance for every trade, should I store price data in a csv? Or would that be unnecessary? Depends on speed.

trades = pd.read_csv('trades.csv')

results = []

for row_name, row in trades.iterrows():
    results.append(actual_outcome(row['ticker'], row['entry_date'], row['entry_price'], row['stop_price'], row['exit_date'], row['exit_price']))
    results.append(initial_stop(row['ticker'], row['entry_date'], row['entry_price'], row['stop_price']))
    results.append(no_stop(row['ticker'], row['entry_date'], row['entry_price'], row['stop_price']))
    


In [46]:
# List of nested dicts converted into dataframe.

results_df = pd.DataFrame(results)
results_df

,entry_date,ticker,entry_price,stop_price,exit_date,exit_price,realised_r,max_r,stop_type
0,2025-08-12,TSLA,340.84,320.15,2025-08-20,320.05,-1.00,0.39,Actual Outcome
1,2025-08-12,TSLA,340.84,320.15,2025-08-20,320.15,-1.00,0.39,Initial Stop
2,2025-08-12,TSLA,340.84,320.15,NaN,NaN,NaN,7.64,No Stop
3,2025-08-26,STX,165.36,151.31,2025-11-21,229.72,4.58,9.34,Actual Outcome
4,2025-08-26,STX,165.36,151.31,NaN,NaN,NaN,29.94,Initial Stop
...,...,...,...,...,...,...,...,...,...
94,2026-04-08,LITE,896.23,761.51,NaN,NaN,NaN,0.47,Initial Stop
95,2026-04-08,LITE,896.23,761.51,NaN,NaN,NaN,0.47,No Stop
96,2026-04-08,STX,495.76,436.84,NaN,NaN,NaN,1.53,Actual Outcome
97,2026-04-08,STX,495.76,436.84,NaN,NaN,NaN,1.53,Initial Stop


In [23]:
results_df.to_csv('results.csv', index=False)